# 🔀 RL-Diffing Crosscoder — Qwen3.5-4B base vs mechreward-G3 (papergrade)

**Companion to `17b_crosscoder_model_diff_papergrade.ipynb`.** Notebook 17b did *instruct-tuning diffing* (Gemma-2-2B base vs IT). This notebook does *RL-diffing*: same base model, but one copy is the LoRA-adapted version that we trained with mech-reward GRPO to break the GSM8K 64% ceiling to 83%.

Together, 17b + 17c are the **two-pair main table** of Paper 1 in the openinterp.org crosscoder series. They isolate two different fine-tuning axes (broad SFT/instruct vs targeted-task RL) on disjoint architectures (Gemma-2-2B dense softmax vs Qwen3.5-4B hybrid GDN).

### What we change vs 17b

| | 17b (instruct) | **17c (RL)** |
|---|---|---|
| Model A | `google/gemma-2-2b` | `Qwen/Qwen3.5-4B` |
| Model B | `google/gemma-2-2b-it` | `caiovicentino1/Qwen3.5-4B-mechreward-G3-phaseA-step400` (LoRA on Qwen3.5-4B) |
| Loading | 2 separate models | **1 base + LoRA toggle via `disable_adapter()`** — saves ~9 GB GPU |
| Architecture | Dense softmax-attention, 26 layers | **Hybrid GDN + full-attn, 32 layers** |
| Layer | 13 (mid-stack) | **18** (downstream of reasoning peak L13, where the mech-reward SAE lives) |
| Data mix | FineWeb-Edu + UltraChat | **FineWeb-Edu + GSM8K** (math is what RL trained on) |
| Tokenizer | Gemma | Qwen3 (matches both since LoRA inherits) |
| Architecture novelty | first BatchTopK crosscoder + causal validation on Gemma | **first cross-stage RL crosscoder + causal validation on hybrid GDN** |

### What's new methodologically

1. **LoRA toggle activation collection.** PEFT's `with model.disable_adapter():` lets us run base and adapted forward passes on the *same* loaded weights. This is much more memory-efficient than loading two copies and (importantly) guarantees identical tokenizer/positional-embedding behaviour.
2. **Math-conditional probe set.** RL targeted GSM8K. The causal validation should sample probe inputs that *include* math, not just generic web text.
3. **The interesting hypothesis.** The mech-reward training used L18 SAE features as a per-token reward. That *should* preserve the semantic alignment of those features at L18 — but maybe RL *causally rewired downstream consumers* of those features even while keeping their decoder direction aligned. If true, we expect Pearson_CE < cosine specifically on shared features that received reward signal — exactly the regime cosine-only universality claims would miss.

### References

- 17b notebook (this notebook's structural twin)
- mechreward G3 result: `caiovicentino/mechreward` repo, GSM8K 64%→83%, LessWrong post 2026-04-17
- Lindsey et al. 2024 — Sparse Crosscoders <https://transformer-circuits.pub/2024/crosscoders/>
- Minder et al. 2025 — Robustly identifying concepts via crosscoders <https://arxiv.org/abs/2504.02922>
- Goodfire RLFR — features-as-reward in dense Gemma
- Stage-Wise Model Diffing (Anthropic 2024) — single-model diffing across training stages, methodologically adjacent

Estimated cost: 1 × A100 night, ~4-5 h training + 30 min validation + 30 min causal eval ≈ **$60–80**.

In [ ]:
# Install — same recipe as 17b plus PEFT for LoRA, plus FLA stack for hybrid GDN speed.
!pip -q install --upgrade transformers accelerate peft torchao safetensors huggingface_hub datasets einops tqdm matplotlib scipy
# Optional but strongly recommended on Qwen3.5: 10x faster GDN forward.
# Skip if causes install pain — fallback torch impl works, just slower.
!pip -q install flash-linear-attention causal-conv1d 2>/dev/null || echo 'FLA install skipped — using torch fallback'

## 1. Configuration

Defaults align with 17b for fair cross-pair comparison. Layer 18 chosen because (a) it sits downstream of the GSM8K reasoning peak at L13, (b) it's the layer where the mech-reward SAE was trained — so we know features there carry reasoning signal.

In [ ]:
import os, math, json, time
from pathlib import Path

CFG = {
    # --- model + adapter pair ---
    'base_model':   'Qwen/Qwen3.5-4B',
    'lora_repo':    'caiovicentino1/Qwen3.5-4B-mechreward-G3-phaseA-step400',
    'layer':        18,                  # downstream of reasoning peak L13; where mech-reward SAE lives
    'd_model':      None,                # auto-detect from model.config.hidden_size

    # --- crosscoder dictionary (same hyperparams as 17b for fair cross-pair comparison) ---
    'expansion':    32,                  # n_features = expansion * d_model
    'k_batchtopk':  100,
    'k_warmup_init': 100,                # NO anneal — Minder default for ≤4B models
    'k_warmup_steps': 1,
    'dec_init_norm': 1.0,                # critical for BatchTopK convergence

    # --- training ---
    'token_budget':  100_000_000,
    'seq_len':       512,
    'fwd_batch':     4,
    'cc_batch':      4096,
    'lr':            1e-4,
    'lambda_l1':     4.1e-2,
    'warmup_steps':  1000,
    'grad_clip':     1.0,
    'lr_decay_frac': 0.20,
    'checkpoint_every_tokens': 5_000_000,

    # --- data mix (math-tilted because RL trained on GSM8K) ---
    'data_web_frac':  0.5,
    'data_math_frac': 0.5,

    # --- publishing ---
    'hf_user':      os.environ.get('HF_USERNAME', 'caiovicentino1'),
    'hf_repo_name': 'qwen3.5-4b-crosscoder-rl-diff-papergrade',
}
CFG['hf_repo'] = f"{CFG['hf_user']}/{CFG['hf_repo_name']}"

LOCAL_OUT = Path('/content/crosscoder_rldiff_out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

print(f"Pair          : {CFG['base_model']}  +  LoRA {CFG['lora_repo']}")
print(f"Layer         : {CFG['layer']}")
print(f"BatchTopK     : k = {CFG['k_batchtopk']} (no anneal)")
print(f"Tokens        : {CFG['token_budget']:,}")
print(f"HF repo       : {CFG['hf_repo']}")

## 2. Load Qwen3.5-4B base + apply LoRA adapter

Single base model in bf16. LoRA loaded via PEFT but kept *toggleable* — for activation collection we run the same model twice per probe, once with `disable_adapter()` (= base activations) and once with adapter active (= RL-adapted activations).

**Memory budget**: Qwen3.5-4B bf16 ≈ 8 GB · LoRA adapter ≈ 50 MB · crosscoder ≈ 3.4 GB · Adam state ≈ 6.8 GB · activation buffer ≈ 5 GB → ~25 GB total. Fits A100-40 comfortably; tight on L4-24, won't fit T4-16.

Per `feedback_mechreward_grpo_infra.md`: Qwen3.5 is multimodal — must use `AutoModelForImageTextToText`, not `CausalLM`. Vision tower stays frozen and irrelevant for residual-stream interp.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login, HfApi, create_repo, hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (write scope): ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Crosscoder training wants a GPU.'

print(f'Loading base {CFG["base_model"]} ...')
tok = AutoTokenizer.from_pretrained(CFG['base_model'], trust_remote_code=True)
base = AutoModelForImageTextToText.from_pretrained(
    CFG['base_model'],
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map={'': device},
    trust_remote_code=True,
)
base.eval()
for p in base.parameters():
    p.requires_grad_(False)

print(f'Attaching LoRA {CFG["lora_repo"]} ...')
model = PeftModel.from_pretrained(base, CFG['lora_repo'], torch_dtype=torch.bfloat16)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Auto-detect d_model + locate transformer block list
def _block_list(m):
    candidates = [m]
    if hasattr(m, 'base_model'):                                    # PEFT wrapper
        candidates.append(m.base_model.model if hasattr(m.base_model, 'model') else m.base_model)
    if hasattr(m, 'model'):
        candidates.append(m.model)
    for s in candidates:
        for path in [('model','language_model','layers'), ('language_model','layers'),
                     ('model','layers'), ('layers',)]:
            cur = s; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('Could not locate layer list — inspect state_dict keys.')

blocks = _block_list(model)
L = CFG['layer']
assert 0 <= L < len(blocks), f'Layer {L} out of range (0..{len(blocks)-1})'

# d_model from base model config
d_model = base.config.text_config.hidden_size if hasattr(base.config, 'text_config') else base.config.hidden_size
CFG['d_model']   = int(d_model)
CFG['n_features'] = CFG['expansion'] * CFG['d_model']
print(f'\nLoaded. d_model = {d_model}, layers = {len(blocks)}, hooking L{L}.')
print(f'Crosscoder dictionary: {CFG["n_features"]:,} latents (expansion {CFG["expansion"]}).')

## 3. Paired activation streamer — LoRA toggle pattern

For each input we run the model **twice**: once with adapter disabled (= base activations, A) and once with adapter active (= RL-adapted activations, B). Single hook captures both, in the right order. Same input string → same token IDs → matched residuals.

Data mix: FineWeb-Edu (general) + GSM8K (math, what RL trained on) at 50/50. Math data is essential — without it the RL-induced features are out-of-distribution during training and we get a degenerate "all shared" crosscoder.

In [ ]:
from datasets import load_dataset
import random

def _web_iter():
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    for row in ds:
        t = row.get('text', '')
        if t and len(t) > 200:
            yield t

def _math_iter():
    # GSM8K is open + small; loop indefinitely
    ds = load_dataset('openai/gsm8k', 'main', split='train')
    while True:
        for row in ds:
            q = row.get('question', '')
            a = row.get('answer', '')
            if q and a:
                yield f"Question: {q}\nAnswer: {a}"

def text_stream(web_frac=CFG['data_web_frac']):
    web, math_it = _web_iter(), _math_iter()
    while True:
        if random.random() < web_frac:
            yield next(web)
        else:
            yield next(math_it)

class SingleHook:
    def __init__(self, blocks, layer):
        self.buf = None
        self.h = blocks[layer].register_forward_hook(self._hook)
    def _hook(self, _mod, _inp, out):
        h = out[0] if isinstance(out, tuple) else out
        self.buf = h.detach()
    def pop(self):
        b = self.buf; self.buf = None; return b
    def close(self):
        self.h.remove()

hook = SingleHook(blocks, L)

@torch.no_grad()
def collect_paired_batch(texts):
    """Run model twice (LoRA off → on), capture residual at L for each, return matched."""
    enc = tok(texts, return_tensors='pt', max_length=CFG['seq_len'],
              truncation=True, padding='max_length').to(device)
    ids, mask = enc['input_ids'], enc['attention_mask']
    # base (LoRA disabled)
    with model.disable_adapter():
        model(ids, attention_mask=mask)
    hA = hook.pop()                                          # (B, T, D) bf16
    # RL-adapted
    model(ids, attention_mask=mask)
    hB = hook.pop()
    # drop BOS, mask padding
    hA, hB = hA[:, 1:, :], hB[:, 1:, :]
    m = mask[:, 1:].bool()
    hA = hA[m].float()
    hB = hB[m].float()
    return hA, hB

# Estimate per-source norm scale
print('Estimating per-source activation-norm scale (100 batches) ...')
txt = text_stream()
normA_acc, normB_acc, n_acc = 0.0, 0.0, 0
for _ in range(100):
    batch = [next(txt) for _ in range(CFG['fwd_batch'])]
    hA, hB = collect_paired_batch(batch)
    normA_acc += hA.norm(dim=-1).mean().item() * hA.shape[0]
    normB_acc += hB.norm(dim=-1).mean().item() * hB.shape[0]
    n_acc += hA.shape[0]
mean_norm_A = normA_acc / n_acc
mean_norm_B = normB_acc / n_acc
norm_scale_A = math.sqrt(CFG['d_model']) / mean_norm_A
norm_scale_B = math.sqrt(CFG['d_model']) / mean_norm_B
CFG['norm_scale_A'] = norm_scale_A
CFG['norm_scale_B'] = norm_scale_B
print(f'Mean ||h_base|| = {mean_norm_A:.3f}  -> scale {norm_scale_A:.4f}')
print(f'Mean ||h_LoRA|| = {mean_norm_B:.3f}  -> scale {norm_scale_B:.4f}')
print(f'Δ||h|| = {(mean_norm_B - mean_norm_A):.3f}  ({100*(mean_norm_B/mean_norm_A - 1):+.2f}%)  '
      f'-- LoRA shifted residual norm; this is the first signal of RL effect at L{L}')

In [ ]:
def paired_stream(buffer_mult=128):
    """Yields (cc_batch, 2, D) float32 minibatches forever, normalized + shuffled."""
    txt = text_stream()
    pending_A, pending_B = [], []
    while True:
        target_rows = CFG['cc_batch'] * buffer_mult
        rows = sum(p.shape[0] for p in pending_A)
        while rows < target_rows:
            batch = [next(txt) for _ in range(CFG['fwd_batch'])]
            hA, hB = collect_paired_batch(batch)
            pending_A.append((hA * norm_scale_A).cpu())
            pending_B.append((hB * norm_scale_B).cpu())
            rows += hA.shape[0]
        all_A = torch.cat(pending_A, dim=0)
        all_B = torch.cat(pending_B, dim=0)
        perm  = torch.randperm(all_A.shape[0])
        all_A = all_A[perm]; all_B = all_B[perm]
        for i in range(0, target_rows - CFG['cc_batch'] + 1, CFG['cc_batch']):
            chunkA = all_A[i:i+CFG['cc_batch']].to(device, non_blocking=True)
            chunkB = all_B[i:i+CFG['cc_batch']].to(device, non_blocking=True)
            yield torch.stack([chunkA, chunkB], dim=1)
        leftover = all_A.shape[0] - (target_rows // CFG['cc_batch']) * CFG['cc_batch']
        if leftover > 0:
            pending_A = [all_A[-leftover:]]
            pending_B = [all_B[-leftover:]]
        else:
            pending_A, pending_B = [], []

print('Paired stream ready.')

## 4. BatchTopK Cross-Stage Crosscoder

Identical architecture to 17b: per-source encoder/decoder, shared `b_enc`, per-source `b_dec`, BatchTopK at training, JumpReLU at inference. Re-using the proven recipe so the two papers are directly comparable in the main table.

In [ ]:
from einops import einsum, rearrange

class BatchTopKCrossCoder(nn.Module):
    def __init__(self, d_model: int, n_features: int, k: int, dec_init_norm: float = 1.0):
        super().__init__()
        self.D, self.N, self.k = d_model, n_features, k
        self.W_enc = nn.Parameter(torch.empty(2, d_model, n_features, dtype=torch.float32))
        self.W_dec = nn.Parameter(torch.empty(n_features, 2, d_model, dtype=torch.float32))
        self.b_enc = nn.Parameter(torch.zeros(n_features, dtype=torch.float32))
        self.b_dec = nn.Parameter(torch.zeros(2, d_model, dtype=torch.float32))
        self.register_buffer('threshold', torch.zeros(n_features, dtype=torch.float32))
        self._inference_mode = False
        with torch.no_grad():
            W = torch.randn_like(self.W_dec)
            W = W / W.norm(dim=-1, keepdim=True).clamp_min(1e-8) * dec_init_norm
            self.W_dec.copy_(W)
            self.W_enc.copy_(rearrange(self.W_dec, 'n m d -> m d n'))

    def encode_pre(self, h):
        h_centered = h - self.b_dec[None]
        return einsum(h_centered, self.W_enc, 'b m d, m d n -> b n') + self.b_enc

    def decoder_norms(self):
        return self.W_dec.norm(dim=-1)

    def encode(self, h, k_now=None):
        pre = self.encode_pre(h)
        acts = F.relu(pre)
        if self._inference_mode:
            return torch.where(pre > self.threshold[None], acts, torch.zeros_like(acts))
        dnorm = self.decoder_norms().sum(dim=-1)
        scaled = acts * dnorm[None]
        k = k_now if k_now is not None else self.k
        topn = k * acts.shape[0]
        flat = scaled.flatten()
        if topn >= flat.numel():
            return acts
        thr = flat.topk(topn, sorted=False).values.min()
        return acts * (scaled >= thr)

    def decode(self, z):
        return einsum(z, self.W_dec, 'b n, n m d -> b m d') + self.b_dec[None]

    def forward(self, h, k_now=None):
        z = self.encode(h, k_now=k_now)
        return self.decode(z), z

    @torch.no_grad()
    def calibrate_threshold(self, stream, n_batches=20):
        was_train = self.training
        self.eval(); self._inference_mode = False
        thrs = []
        for _ in range(n_batches):
            h = next(stream)
            pre = self.encode_pre(h)
            z = self.encode(h)
            survived = pre * (z > 0).float()
            survived[survived == 0] = float('inf')
            thrs.append(survived.min(dim=0).values)
        thr = torch.stack(thrs, dim=0).min(dim=0).values
        thr = torch.where(torch.isfinite(thr), thr, torch.zeros_like(thr))
        self.threshold.copy_(thr.clamp_min(0))
        self._inference_mode = True
        if was_train: self.train()

cc = BatchTopKCrossCoder(
    d_model=CFG['d_model'], n_features=CFG['n_features'],
    k=CFG['k_batchtopk'], dec_init_norm=CFG['dec_init_norm'],
).to(device)
n_params = sum(p.numel() for p in cc.parameters())
print(f'Crosscoder: {CFG["n_features"]:,} latents · {n_params/1e6:.1f}M params · dec_init_norm = {CFG["dec_init_norm"]}')

## 5. Training loop with HF checkpoint resume

Identical schedule + resume logic to 17b. Pushes `crosscoder_resume.pt` to HF every 5M tokens. Kernel crash recovers automatically by re-running this cell.

Expected behaviour (mirroring 17b trajectory): VE negative during warmup (steps 0–1000), jumps positive once LR hits max, plateau at ~0.85–0.90 by 100M tokens.

In [ ]:
from tqdm.auto import tqdm

api = HfApi()
create_repo(CFG['hf_repo'], exist_ok=True, private=False, token=HF_TOKEN)

opt = torch.optim.Adam(cc.parameters(), lr=CFG['lr'], betas=(0.9, 0.999))

def k_at(step):
    if step >= CFG['k_warmup_steps']:
        return CFG['k_batchtopk']
    frac = step / CFG['k_warmup_steps']
    return int(CFG['k_warmup_init'] - frac * (CFG['k_warmup_init'] - CFG['k_batchtopk']))

def lr_at(step, total_steps):
    if step < CFG['warmup_steps']:
        return CFG['lr'] * (step + 1) / CFG['warmup_steps']
    decay_start = int((1 - CFG['lr_decay_frac']) * total_steps)
    if step < decay_start:
        return CFG['lr']
    prog = (step - decay_start) / max(1, total_steps - decay_start)
    return CFG['lr'] * (1.0 - prog)

total_steps = CFG['token_budget'] // CFG['cc_batch']
log = {'step': [], 'loss': [], 'mse_A': [], 'mse_B': [], 'l1_norm': [], 've_A': [], 've_B': [], 'k': [], 'lr': []}
tokens_seen = 0
step0 = 0

try:
    ckpt_path = hf_hub_download(repo_id=CFG['hf_repo'], filename='crosscoder_resume.pt', token=HF_TOKEN)
    state = torch.load(ckpt_path, map_location=device)
    cc.load_state_dict(state['cc'])
    opt.load_state_dict(state['opt'])
    step0 = state['step']
    tokens_seen = state['tokens_seen']
    log = state.get('log', log)
    print(f'Resumed at step {step0:,}, tokens_seen {tokens_seen:,}')
except (HfHubHTTPError, FileNotFoundError):
    print('No resume checkpoint — starting fresh.')

stream = paired_stream()
next_ckpt = tokens_seen + CFG['checkpoint_every_tokens']
lambda_l1 = CFG['lambda_l1']

pbar = tqdm(range(step0, total_steps), dynamic_ncols=True, initial=step0, total=total_steps)
for step in pbar:
    h = next(stream)
    k_now = k_at(step)
    lr_now = lr_at(step, total_steps)
    for g in opt.param_groups:
        g['lr'] = lr_now

    h_hat, z = cc(h, k_now=k_now)
    mse_per_model = (h_hat - h).pow(2).sum(dim=-1).mean(dim=0)
    recon = mse_per_model.sum()
    dnorm = cc.decoder_norms().sum(dim=-1)
    l1 = (z.abs() * dnorm[None]).sum(dim=-1).mean()
    loss = recon + lambda_l1 * l1

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cc.parameters(), CFG['grad_clip'])
    opt.step()
    tokens_seen += CFG['cc_batch']

    if step % 50 == 0:
        with torch.no_grad():
            var_pm = h.var(dim=(0, 2)).clamp_min(1e-8)
            res_pm = (h_hat - h).var(dim=(0, 2))
            ve = (1 - res_pm / var_pm).tolist()
        log['step'].append(step)
        log['loss'].append(float(loss.item()))
        log['mse_A'].append(float(mse_per_model[0].item()))
        log['mse_B'].append(float(mse_per_model[1].item()))
        log['l1_norm'].append(float(l1.item()))
        log['ve_A'].append(ve[0]); log['ve_B'].append(ve[1])
        log['k'].append(k_now); log['lr'].append(lr_now)
        pbar.set_postfix({
            'loss': f'{loss.item():.2f}',
            'VE_base': f'{ve[0]:.3f}', 'VE_LoRA': f'{ve[1]:.3f}',
            'k': k_now, 'tok': f'{tokens_seen/1e6:.1f}M',
        })

    if tokens_seen >= next_ckpt:
        torch.save({
            'cc': cc.state_dict(), 'opt': opt.state_dict(),
            'step': step + 1, 'tokens_seen': tokens_seen, 'log': log, 'cfg': CFG,
        }, str(LOCAL_OUT / 'crosscoder_resume.pt'))
        try:
            api.upload_file(
                path_or_fileobj=str(LOCAL_OUT / 'crosscoder_resume.pt'),
                path_in_repo='crosscoder_resume.pt',
                repo_id=CFG['hf_repo'], repo_type='model', token=HF_TOKEN,
            )
        except Exception as e:
            print(f'Resume upload failed (continuing): {e}')
        next_ckpt += CFG['checkpoint_every_tokens']

cc.eval()
cc.calibrate_threshold(paired_stream(buffer_mult=8), n_batches=20)
print('Training done. JumpReLU thresholds calibrated.')

## 6. Validation + Δ_norm taxonomy

Per-source variance-explained, L0, dead frac, then Δ_norm classification: `base_only` (RL killed this feature) / `shared` (preserved by RL) / `LoRA_only` (RL added this feature) / unclassified / dead.

Compared to 17b: we expect the **dead frac** lower (RL-LoRA delta is much smaller than instruct-tuning delta) and possibly **fewer LoRA_only** features (RL targeted GSM8K, narrow scope).

**The interesting question** for paper-1: does the Δ_norm distribution at L18 (where mech-reward SAE features were used as RL reward) tilt MORE toward `shared` than at other layers? If yes → reward-aligned features are decoder-preserved. If no → RL silently added/killed features at the reward-targeted layer too.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

val_stream = paired_stream(buffer_mult=8)
N_VAL_BATCHES = 20
n_feat = CFG['n_features']

ve_A_acc, ve_B_acc, l0_acc, fired = [], [], [], torch.zeros(n_feat, device=device)
with torch.no_grad():
    for _ in tqdm(range(N_VAL_BATCHES), desc='validation'):
        h = next(val_stream)
        h_hat, z = cc(h)
        var_pm = h.var(dim=(0, 2)).clamp_min(1e-8)
        res_pm = (h_hat - h).var(dim=(0, 2))
        ve_A_acc.append((1 - res_pm[0] / var_pm[0]).item())
        ve_B_acc.append((1 - res_pm[1] / var_pm[1]).item())
        l0_acc.append((z > 0).float().sum(dim=-1).mean().item())
        fired += (z > 0).float().sum(dim=0)

ve_A = float(np.mean(ve_A_acc)); ve_B = float(np.mean(ve_B_acc))
l0 = float(np.mean(l0_acc))
dead_frac = float((fired == 0).float().mean().item())
print(f'VE (base) = {ve_A:.4f}    VE (LoRA) = {ve_B:.4f}')
print(f'L0         = {l0:.1f}')
print(f'Dead frac  = {dead_frac*100:.2f}%')

with torch.no_grad():
    dn = cc.decoder_norms().detach()
norm_A = dn[:, 0].cpu().numpy()
norm_B = dn[:, 1].cpu().numpy()
delta_norm = 0.5 * (1 + (norm_B - norm_A) / np.maximum(norm_A, norm_B).clip(min=1e-8))

label = np.full(n_feat, 'unclassified', dtype=object)
label[delta_norm <= 0.1] = 'base_only'        # RL killed this feature
label[delta_norm >= 0.9] = 'LoRA_only'        # RL added this feature
label[(delta_norm >= 0.4) & (delta_norm <= 0.6)] = 'shared'
fired_np = fired.cpu().numpy()
label[fired_np == 0] = 'dead'

from collections import Counter
ctr = Counter(label.tolist())
print('\nFeature taxonomy (RL-diffing axis):')
for k in ['base_only', 'shared', 'LoRA_only', 'unclassified', 'dead']:
    pct = 100 * ctr.get(k, 0) / n_feat
    print(f'  {k:14s} {ctr.get(k,0):>6,}   ({pct:5.2f}%)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(delta_norm[fired_np > 0], bins=80, color='#1f77b4', alpha=0.85)
ax.axvspan(0.0, 0.1, alpha=0.15, color='#ff7f0e', label='base_only (RL killed)')
ax.axvspan(0.4, 0.6, alpha=0.15, color='#2ca02c', label='shared (RL preserved)')
ax.axvspan(0.9, 1.0, alpha=0.15, color='#d62728', label='LoRA_only (RL added)')
ax.set_xlabel(r'$\Delta_{\mathrm{norm}}$')
ax.set_ylabel('# features')
ax.set_title(f'RL-Diffing taxonomy — Qwen3.5-4B base vs mechreward-G3, L{CFG["layer"]}')
ax.legend(loc='upper center')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(LOCAL_OUT / 'delta_norm_hist.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Top-activating examples per class

For each class (`base_only`, `shared`, `LoRA_only`), show top-firing tokens. Expect to see:
- `base_only`: features the RL-LoRA *removed*. May correspond to behaviours penalized during GRPO (e.g., format errors, premature termination, off-topic continuation).
- `shared`: features both copies use. Should include math vocabulary and reasoning structure.
- `LoRA_only`: features RL *introduced*. The smoking gun — what did mech-reward GRPO actually teach?

In [ ]:
TOP_PER_CLASS = 5; K_EXAMPLES = 6; PROBE_BATCHES = 10

probe_tokens, probe_z = [], []
txt = text_stream()
with torch.no_grad():
    for _ in tqdm(range(PROBE_BATCHES), desc='probe collection'):
        batch = [next(txt) for _ in range(CFG['fwd_batch'])]
        enc = tok(batch, return_tensors='pt', max_length=CFG['seq_len'],
                  truncation=True, padding='max_length').to(device)
        ids, mask = enc['input_ids'], enc['attention_mask']
        with model.disable_adapter():
            model(ids, attention_mask=mask)
        hA = hook.pop()[:, 1:, :]
        model(ids, attention_mask=mask)
        hB = hook.pop()[:, 1:, :]
        m = mask[:, 1:].bool()
        hA = (hA[m].float() * norm_scale_A)
        hB = (hB[m].float() * norm_scale_B)
        h = torch.stack([hA, hB], dim=1)
        z = cc.encode(h)
        for bi in range(ids.shape[0]):
            valid = mask[bi, 1:].nonzero(as_tuple=True)[0]
            for pos in valid.tolist():
                probe_tokens.append(tok.decode(ids[bi, pos+1].item()))
        probe_z.append(z.cpu())
probe_z = torch.cat(probe_z, dim=0).numpy()
print(f'Probe set: {len(probe_tokens):,} tokens')

def show_top_for_class(cls, n_features=TOP_PER_CLASS, n_examples=K_EXAMPLES):
    feats = np.where(label == cls)[0]
    if len(feats) == 0:
        print(f'  [{cls}] (none)')
        return
    by_total = np.argsort(-np.abs(probe_z[:, feats]).sum(axis=0))[:n_features]
    print(f'\n=== {cls.upper()} ({len(feats)} features total) ===')
    for fi in by_total:
        f = feats[fi]
        z_f = probe_z[:, f]
        top = np.argsort(-np.abs(z_f))[:n_examples]
        toks = [(probe_tokens[t][:30].replace('\n', '\\n'), z_f[t]) for t in top]
        rendered = '  '.join(f'{t!r}({z:+.2f})' for t, z in toks)
        print(f'  f{f:6d} | Δ={delta_norm[f]:.2f} | ‖base‖={norm_A[f]:.2f} ‖LoRA‖={norm_B[f]:.2f}')
        print(f'         {rendered}')

show_top_for_class('LoRA_only')
show_top_for_class('shared')
show_top_for_class('base_only')

## 8. ⭐ Causal validation — Pearson_CE on RL-diff pair

Same protocol as 17b §8. The hypothesis specific to RL-diffing:

> *Mechreward GRPO trained features at L18 to be reward-aligned. Decoder cosine should look very high (RL didn't move directions much). But Pearson_CE should reveal whether RL silently rewired downstream consumers of those features — "causally rewired but cosine unchanged".*

If we see **median cosine > median Pearson_CE** by a large gap on `shared` features (mirroring 17b), it's a second pair confirming the cosine-vs-causal divergence. If the gap is *larger* than 17b's, RL specifically induces causal rewiring without representational displacement — a striking finding for the alignment community.

In [ ]:
from scipy.stats import pearsonr

CAUSAL_PER_CLASS    = 80
CAUSAL_CONTROL      = 30
CAUSAL_PROBE_INPUTS = 256
MIN_FIRINGS         = 3

total_val_tokens = 20 * CFG['cc_batch']
fire_rate = fired_np / total_val_tokens
shared_all  = np.where(label == 'shared')[0]
control_all = np.where(label == 'unclassified')[0]
high_shared  = shared_all[fire_rate[shared_all] > 0.01]
high_control = control_all[fire_rate[control_all] > 0.01]
print(f'Shared firing >1%:  {len(high_shared):>6,} / {len(shared_all):>6,}')
print(f'Control firing >1%: {len(high_control):>6,} / {len(control_all):>6,}')

shared_idx  = np.random.RandomState(0).choice(high_shared,  min(CAUSAL_PER_CLASS, len(high_shared)),  replace=False)
control_idx = np.random.RandomState(1).choice(high_control, min(CAUSAL_CONTROL,   len(high_control)), replace=False)
test_idx = np.concatenate([shared_idx, control_idx])
print(f'Testing {len(shared_idx)} shared + {len(control_idx)} control = {len(test_idx)} features.')

# --- collect 256 probes; baseline logits for both base and LoRA ---
probe_ids, probe_pos = [], []
probe_baseline_logits_A, probe_baseline_logits_B = [], []
txt = text_stream()
with torch.no_grad():
    while len(probe_ids) < CAUSAL_PROBE_INPUTS:
        batch = [next(txt) for _ in range(CFG['fwd_batch'])]
        enc = tok(batch, return_tensors='pt', max_length=128, truncation=True, padding='max_length').to(device)
        for bi in range(enc['input_ids'].shape[0]):
            n_valid = enc['attention_mask'][bi].sum().item()
            if n_valid < 32:
                continue
            ids = enc['input_ids'][bi:bi+1, :n_valid]
            pos = n_valid - 1
            with model.disable_adapter():
                outA = model(ids).logits[0, pos].float()
            outB = model(ids).logits[0, pos].float()
            probe_ids.append(ids.cpu()); probe_pos.append(pos)
            probe_baseline_logits_A.append(outA.cpu())
            probe_baseline_logits_B.append(outB.cpu())
            if len(probe_ids) >= CAUSAL_PROBE_INPUTS:
                break
print(f'Collected {len(probe_ids)} probe inputs.')

# --- ablation hook (subtract f * d at last token of layer L) ---
class AblationHook:
    def __init__(self, blocks, layer, pos):
        self.layer = layer; self.pos = pos
        self.vec = None
        self.h = blocks[layer].register_forward_hook(self._hook)
    def set_vec(self, v):
        self.vec = v
    def _hook(self, _mod, _inp, out):
        if self.vec is None:
            return out
        h = out[0] if isinstance(out, tuple) else out
        h = h.clone()
        h[:, self.pos, :] = h[:, self.pos, :] - self.vec.to(h.dtype).to(h.device)
        return (h,) + out[1:] if isinstance(out, tuple) else h
    def close(self):
        self.h.remove()

def kl(p_logits, q_logits):
    p = F.log_softmax(p_logits.float(), dim=-1)
    q = F.log_softmax(q_logits.float(), dim=-1)
    return (p.exp() * (p - q)).sum().item()

ce_records = []
firing_counts = []
for f_id in tqdm(test_idx, desc='causal validation (RL-diff)'):
    eA, eB = [], []
    d_A_unscaled = cc.W_dec[f_id, 0].detach() / norm_scale_A
    d_B_unscaled = cc.W_dec[f_id, 1].detach() / norm_scale_B
    for ids, pos, bl_A, bl_B in zip(probe_ids, probe_pos, probe_baseline_logits_A, probe_baseline_logits_B):
        ids = ids.to(device)
        # encode pair to get f_val
        with torch.no_grad():
            with model.disable_adapter():
                model(ids); hA = hook.pop()[0, pos].float() * norm_scale_A
            model(ids); hB = hook.pop()[0, pos].float() * norm_scale_B
            h_pair = torch.stack([hA, hB], dim=0).unsqueeze(0)
            z = cc.encode(h_pair)[0]
        f_val = z[f_id].item()
        if f_val <= 0:
            continue
        # ablate in base (LoRA disabled)
        ablA = AblationHook(blocks, L, pos); ablA.set_vec(f_val * d_A_unscaled)
        with torch.no_grad():
            with model.disable_adapter():
                tlA = model(ids).logits[0, pos].float().cpu()
        ablA.close()
        # ablate in LoRA
        ablB = AblationHook(blocks, L, pos); ablB.set_vec(f_val * d_B_unscaled)
        with torch.no_grad():
            tlB = model(ids).logits[0, pos].float().cpu()
        ablB.close()
        eA.append(kl(tlA, bl_A))
        eB.append(kl(tlB, bl_B))
    firing_counts.append(len(eA))
    if len(eA) >= MIN_FIRINGS:
        if len(set(eA)) > 1 and len(set(eB)) > 1:
            r, _ = pearsonr(eA, eB)
        else:
            r = 0.0
        ce_records.append({
            'feature': int(f_id),
            'class': str(label[f_id]),
            'delta_norm': float(delta_norm[f_id]),
            'cosine': float(F.cosine_similarity(
                cc.W_dec[f_id, 0].detach().unsqueeze(0),
                cc.W_dec[f_id, 1].detach().unsqueeze(0)).item()),
            'mean_KL_A': float(np.mean(eA)),
            'mean_KL_B': float(np.mean(eB)),
            'pearson_CE': float(r),
            'n_probes_fired': int(len(eA)),
        })

print(f'\nFiring stats: median={np.median(firing_counts):.1f}, '
      f'n>={MIN_FIRINGS}: {sum(1 for f in firing_counts if f >= MIN_FIRINGS)}/{len(firing_counts)}')

import pandas as pd
df = pd.DataFrame(ce_records)
if len(df):
    print('\nCausal-equivalence summary (RL-diff):')
    print(df.groupby('class')['pearson_CE'].describe()[['count', 'mean', '50%', 'min', 'max']])
    df.to_csv(LOCAL_OUT / 'causal_validation.csv', index=False)
else:
    print('No records — try CAUSAL_PROBE_INPUTS=512 or MIN_FIRINGS=2.')

## 9. Cosine vs Causal scatter — RL-diff edition

Same scatter as 17b §9 but for the RL pair. The killer figure for the second row of the paper's main table.

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = {'shared': '#2ca02c', 'unclassified': '#7f7f7f',
            'LoRA_only': '#d62728', 'base_only': '#ff7f0e'}
    for cls, sub in df.groupby('class'):
        ax.scatter(sub['cosine'], sub['pearson_CE'], s=28, alpha=0.65,
                   color=cmap.get(cls, '#1f77b4'), label=f'{cls} (n={len(sub)})')
    ax.plot([-1, 1], [-1, 1], '--', color='k', alpha=0.4, label='cosine = CE')
    ax.axhline(0.5, ls=':', color='r', alpha=0.5)
    ax.axvline(0.5, ls=':', color='r', alpha=0.5)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel(r'Decoder cosine $\cos(d^{\rm base}_j, d^{\rm LoRA}_j)$')
    ax.set_ylabel(r'Pearson CE  (causal effect correlation)')
    ax.set_title('Cosine vs causal universality — Qwen3.5-4B base vs mechreward-G3\n(RL-diffing edition)')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(LOCAL_OUT / 'cosine_vs_causal.png', dpi=200, bbox_inches='tight')
    fig.savefig(LOCAL_OUT / 'cosine_vs_causal.pdf', bbox_inches='tight')
    plt.show()

    shared = df[df['class'] == 'shared']
    if len(shared):
        gap_frac = float(((shared['cosine'] > 0.7) & (shared['pearson_CE'] < 0.5)).mean())
        print(f'\nFraction of shared with cosine>0.7 AND CE<0.5 (RL pair): {gap_frac:.2%}')
        print(f'Median CE (shared, RL pair):       {shared["pearson_CE"].median():.3f}')
        print(f'Median cosine (shared, RL pair):   {shared["cosine"].median():.3f}')
    if 'unclassified' in df['class'].unique():
        ctrl = df[df['class'] == 'unclassified']
        print(f'Median CE (control, RL pair):      {ctrl["pearson_CE"].median():.3f}')
    print('\n--- Compare against 17b (Gemma-2-2B base/IT, instruct-diffing) ---')
    print('  Median cosine (shared):            0.965')
    print('  Median CE (shared):                0.616')
    print('  Fraction cosine>0.7 AND CE<0.5:    38.24%')

## 10. Save artifacts to HF

Same layout as 17b. Final repo will be the second public RL-diffing crosscoder with paired causal validation.

In [ ]:
from safetensors.torch import save_file

save_file({
    'W_enc': cc.W_enc.detach().cpu().contiguous(),
    'W_dec': cc.W_dec.detach().cpu().contiguous(),
    'b_enc': cc.b_enc.detach().cpu().contiguous(),
    'b_dec': cc.b_dec.detach().cpu().contiguous(),
    'threshold': cc.threshold.detach().cpu().contiguous(),
}, str(LOCAL_OUT / 'crosscoder_final.safetensors'))

(LOCAL_OUT / 'cfg.json').write_text(json.dumps({
    'architecture': 'cross_stage_crosscoder_lora',
    'variant':      'BatchTopK with JumpReLU inference threshold',
    'reference':    'Companion to 17b (Gemma-2-2B base/IT). Pair-2 of openinterp.org crosscoder Paper-1.',
    **CFG,
    'val_metrics': {'ve_A': ve_A, 've_B': ve_B, 'L0': l0, 'dead_frac': dead_frac},
    'taxonomy_counts': dict(ctr),
}, indent=2))

(LOCAL_OUT / 'feature_taxonomy.json').write_text(json.dumps({
    'thresholds': {'base_only': [0, 0.1], 'shared': [0.4, 0.6], 'LoRA_only': [0.9, 1.0]},
    'features': [
        {
            'feature':    int(j),
            'delta_norm': float(delta_norm[j]),
            'norm_A':     float(norm_A[j]),
            'norm_B':     float(norm_B[j]),
            'fired_count':int(fired_np[j]),
            'label':      str(label[j]),
        } for j in range(n_feat)
    ],
}))

(LOCAL_OUT / 'train_log.json').write_text(json.dumps(log))

readme = f"""---
license: apache-2.0
tags:
  - mechanistic-interpretability
  - sparse-autoencoder
  - crosscoder
  - rl-diffing
  - mechreward
base_model:
  - {CFG['base_model']}
  - {CFG['lora_repo']}
---

# RL-Diffing Crosscoder — Qwen3.5-4B base vs mechreward-G3 (papergrade)

BatchTopK crosscoder trained jointly on base `{CFG['base_model']}` and the
mechreward-G3 LoRA-adapted variant (`{CFG['lora_repo']}`, GSM8K 64%→83%) at
layer {CFG['layer']} residual stream.

Companion to `caiovicentino1/gemma2-2b-crosscoder-model-diff-papergrade`.
Together they form the two-pair main table of openinterp.org Paper 1
(decoder-cosine vs Pearson causal-equivalence in cross-model crosscoders).

## Recipe

- Single base model + LoRA toggle via `peft.PeftModel.disable_adapter()`
- BatchTopK k = {CFG['k_batchtopk']}, expansion {CFG['expansion']} → {CFG['n_features']:,} latents
- {CFG['token_budget']/1e6:.0f} M training tokens (FineWeb-Edu + GSM8K, 50/50)
- Per-source norm scaling, BOS dropped
- Adam lr {CFG['lr']}, warmup {CFG['warmup_steps']}, decay last 20%

## Validation

| | base (A) | LoRA (B) |
|---|---|---|
| variance explained | {ve_A:.4f} | {ve_B:.4f} |

L0 = {l0:.1f}, dead-feature fraction = {dead_frac*100:.2f}%

## Δ_norm taxonomy

{json.dumps(dict(ctr), indent=2)}

## Causal validation

Pearson_CE on each shared feature: ablate in base + LoRA on matched probe inputs
(GSM8K + FineWeb mix, n=256, last-token), correlate the two KL-shift series.
Records in `causal_validation.csv`. Figure `cosine_vs_causal.png`.

## Citation chain

- Lindsey et al. 2024 — Sparse Crosscoders
- Anthropic Jan 2025 — Crosscoder Diffing Update
- Minder et al. NeurIPS 2025 (arxiv:2504.02922)
- mechreward Stage Gate 3 — LessWrong post 2026-04-17
- (this repo) — Pearson_CE applied to RL-diffing

## Reproduce

Notebook: `OpenInterpretability/notebooks/17c_crosscoder_rl_diffing_papergrade.ipynb`
"""
(LOCAL_OUT / 'README.md').write_text(readme)

api.upload_folder(
    folder_path=str(LOCAL_OUT),
    repo_id=CFG['hf_repo'],
    repo_type='model',
    token=HF_TOKEN,
    ignore_patterns=['*.ipynb_checkpoints*', '*resume.pt'],
)
print(f'\n✅ Uploaded to https://huggingface.co/{CFG["hf_repo"]}')

## 11. Two-pair comparison — main table of Paper 1

Pulls numbers from this run + the 17b artifact and prints the comparison table that goes into the paper's Section 3.

In [ ]:
# Pull 17b numbers from its HF repo for the cross-pair table
from huggingface_hub import hf_hub_download

def safe_load_17b():
    try:
        cfg_p = hf_hub_download('caiovicentino1/gemma2-2b-crosscoder-model-diff-papergrade',
                                'cfg.json', token=HF_TOKEN)
        cv_p  = hf_hub_download('caiovicentino1/gemma2-2b-crosscoder-model-diff-papergrade',
                                'causal_validation.csv', token=HF_TOKEN)
        cfg17b = json.load(open(cfg_p))
        df17b  = pd.read_csv(cv_p)
        return cfg17b, df17b
    except Exception as e:
        print(f'(17b pull failed: {e}; main table will only show 17c)')
        return None, None

cfg17b, df17b = safe_load_17b()

def summary(cfg_, df_, pair_name, model_a, model_b):
    if df_ is None or len(df_) == 0:
        return None
    s = df_[df_['class'] == 'shared']
    c = df_[df_['class'] == 'unclassified']
    return {
        'pair':       pair_name,
        'model_A':    model_a,
        'model_B':    model_b,
        'layer':      cfg_['layer'] if cfg_ else None,
        'd_model':    cfg_['d_model'] if cfg_ else None,
        've_A':       cfg_['val_metrics']['ve_A'] if cfg_ else None,
        've_B':       cfg_['val_metrics']['ve_B'] if cfg_ else None,
        'n_shared':   len(s),
        'med_cosine': float(s['cosine'].median()) if len(s) else None,
        'med_CE':     float(s['pearson_CE'].median()) if len(s) else None,
        'gap_frac':   float(((s['cosine'] > 0.7) & (s['pearson_CE'] < 0.5)).mean()) if len(s) else None,
        'med_CE_ctrl':float(c['pearson_CE'].median()) if len(c) else None,
    }

rows = []
if cfg17b is not None:
    rows.append(summary(cfg17b, df17b, 'instruct-diffing',
                        'google/gemma-2-2b', 'google/gemma-2-2b-it'))
with open(LOCAL_OUT / 'cfg.json') as f:
    cfg17c = json.load(f)
rows.append(summary(cfg17c, df, 'RL-diffing',
                    CFG['base_model'], CFG['lora_repo']))

import pandas as pd
tbl = pd.DataFrame([r for r in rows if r is not None])
print('\n=== Paper-1 main table ===\n')
print(tbl.to_string(index=False))
tbl.to_csv(LOCAL_OUT / 'paper1_main_table.csv', index=False)
tbl.to_markdown(LOCAL_OUT / 'paper1_main_table.md', index=False)

try:
    api.upload_file(
        path_or_fileobj=str(LOCAL_OUT / 'paper1_main_table.csv'),
        path_in_repo='paper1_main_table.csv',
        repo_id=CFG['hf_repo'], repo_type='model', token=HF_TOKEN,
    )
    api.upload_file(
        path_or_fileobj=str(LOCAL_OUT / 'paper1_main_table.md'),
        path_in_repo='paper1_main_table.md',
        repo_id=CFG['hf_repo'], repo_type='model', token=HF_TOKEN,
    )
    print(f'\nMain table pushed to https://huggingface.co/{CFG["hf_repo"]}/blob/main/paper1_main_table.md')
except Exception as e:
    print(f'Main-table upload skipped: {e}')

## 12. Where this leaves us

After this notebook runs, Paper 1 has:

1. **Two pairs** in the main table — instruct-diffing (17b) and RL-diffing (17c) — across **two architectures** (Gemma-2-2B dense + Qwen3.5-4B hybrid GDN).
2. **The same metric (Pearson_CE)** applied to both, enabling direct comparison.
3. **The same scatter figure** for each pair, enabling side-by-side display.
4. **Caveats consistent across pairs** — both subject to zero-ablation OOD, single-layer, prefiltered firing rate; declared honestly.

### Concrete predictions for this run

- **VE both ≥ 0.85** (mirroring 17b paper-grade convergence)
- **Dead frac < 17b's 43%** — RL is a smaller delta than instruct-tuning
- **More `shared` features, fewer `LoRA_only`** — RL changed less of the network than full instruct-tuning would
- **Median cosine still ~0.95+** for shared — RL preserved decoder directions
- **Median Pearson_CE ?** — open question. If lower than 17b's 0.616, RL produces *more* causal rewiring per cosine-preserved feature. That's the headline.

### Next steps after both pairs run

1. **Optimal ablation robustness check** (Li & Janson 2024, $30, 2 days). Re-run §8 of both notebooks with optimal-ablation. Show the gap survives. Closes attack #3 in the adversarial review.
2. **Paper-1 draft** (~3–4 days). Workshop short paper, 4 pages. Submit to ICML MI Workshop alongside the hallucination paper.
3. **CLCC follow-up** (Causal-Loss-trained Crosscoder, $500, 2–3 weeks). NeurIPS 2026 main target. The S-tier paper.

### Reading queue

- Lindsey et al. 2024 — <https://transformer-circuits.pub/2024/crosscoders/index.html>
- Anthropic Jan 2025 — <https://transformer-circuits.pub/2025/crosscoder-diffing-update/index.html>
- Minder et al. NeurIPS 2025 — <https://arxiv.org/abs/2504.02922>
- Li & Janson NeurIPS 2024 (Optimal Ablation) — <https://arxiv.org/abs/2409.09951>
- Park et al. (Steering vector non-identifiability) — <https://arxiv.org/abs/2602.06801>
- Geiger et al. JMLR 2025 (Causal abstraction) — <https://arxiv.org/abs/2301.04709>
- Goodfire RLFR (closest neighbor for RL+SAE direction) — <https://www.goodfire.ai/research/rlfr>